In [1]:
!pip install openpyxl -q

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os
import json
import zipfile
import shutil
import random
from pathlib import Path

import pandas as pd
import numpy as np

from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import MobileNet_V3_Small_Weights

# =========================
# CONFIGURACIÓN GENERAL
# =========================

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/ModeloEntrenado")

ZIP_NAME = "insect_parts_dataset_combined.zip"
EXTRACT_PATH = Path("/content/insect_dataset")

FINAL_MODELS_DIR = DRIVE_PROJECT_DIR / "final_models"
FINAL_MODELS_DIR.mkdir(parents=True, exist_ok=True)

MOBILENET_EPOCHS = 15
BATCH_SIZE = 16
IMG_SIZE_MOBILENET = 224

TRAIN_MOBILENET = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print("Dispositivo:", device)
print("Carpeta de proyecto en Drive:", DRIVE_PROJECT_DIR)

Mounted at /content/drive
Dispositivo: cuda
Carpeta de proyecto en Drive: /content/drive/MyDrive/ModeloEntrenado


In [3]:
def find_file(filename, search_roots):
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue

        direct = root / filename
        if direct.exists():
            return direct

        matches = list(root.rglob(filename))
        if matches:
            return matches[0]

    return None


ZIP_PATH = find_file(
    ZIP_NAME,
    [
        DRIVE_PROJECT_DIR,
        Path("/content/drive/MyDrive"),
        Path("/content")
    ]
)

if ZIP_PATH is None:
    raise FileNotFoundError(
        f"No encontré {ZIP_NAME}. "
        "Verifica que esté en la carpeta ModeloEntrenado o súbelo a Colab."
    )

print("ZIP encontrado en:", ZIP_PATH)

if EXTRACT_PATH.exists():
    shutil.rmtree(EXTRACT_PATH)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Dataset descomprimido en:", EXTRACT_PATH)

data_yaml_files = list(EXTRACT_PATH.rglob("data.yaml"))

if not data_yaml_files:
    raise FileNotFoundError("No se encontró data.yaml dentro del ZIP.")

DATA_YAML = data_yaml_files[0]
DATASET_ROOT = DATA_YAML.parent

print("data.yaml:", DATA_YAML)
print("Dataset root:", DATASET_ROOT)

ZIP encontrado en: /content/drive/MyDrive/ModeloEntrenado/insect_parts_dataset_combined.zip
Dataset descomprimido en: /content/insect_dataset
data.yaml: /content/insect_dataset/insect_parts_dataset_combined/data.yaml
Dataset root: /content/insect_dataset/insect_parts_dataset_combined


In [4]:
SPECIES_TO_INSECT = {
    "aedes_albopictus": "asian_tiger_mosquito",
    "apis_mellifera": "honey_bee",
    "danaus_plexippus": "monarch_butterfly",
    "coccinella_septempunctata": "sevenspotted_lady_beetle",
    "dissosteira_carolina": "carolina_grasshopper",
    "mantis_religiosa": "european_mantid"
}

TRAIT_COLUMNS = [
    "wing_count",
    "forewing_type",
    "mouthpart_type",
    "antenna_type",
    "leg_specialization",
    "body_shape",
    "waist_shape"
]

TRAITS_BY_INSECT = {
    "asian_tiger_mosquito": {
        "wing_count": "one_pair",
        "forewing_type": "membranous",
        "mouthpart_type": "piercing_sucking",
        "antenna_type": "filiform",
        "leg_specialization": "generalist",
        "body_shape": "slender",
        "waist_shape": "no_waist"
    },
    "honey_bee": {
        "wing_count": "two_pairs",
        "forewing_type": "membranous",
        "mouthpart_type": "chewing",
        "antenna_type": "filiform",
        "leg_specialization": "generalist",
        "body_shape": "elongated",
        "waist_shape": "narrow_waist"
    },
    "monarch_butterfly": {
        "wing_count": "two_pairs",
        "forewing_type": "scaly",
        "mouthpart_type": "siphoning",
        "antenna_type": "clavate",
        "leg_specialization": "generalist",
        "body_shape": "slender",
        "waist_shape": "no_waist"
    },
    "sevenspotted_lady_beetle": {
        "wing_count": "two_pairs",
        "forewing_type": "elytra",
        "mouthpart_type": "chewing",
        "antenna_type": "clavate",
        "leg_specialization": "generalist",
        "body_shape": "rounded_domed",
        "waist_shape": "no_waist"
    },
    "carolina_grasshopper": {
        "wing_count": "two_pairs",
        "forewing_type": "tegmina",
        "mouthpart_type": "chewing",
        "antenna_type": "filiform",
        "leg_specialization": "jumping",
        "body_shape": "elongated",
        "waist_shape": "no_waist"
    },
    "european_mantid": {
        "wing_count": "two_pairs",
        "forewing_type": "tegmina",
        "mouthpart_type": "chewing",
        "antenna_type": "filiform",
        "leg_specialization": "grasping",
        "body_shape": "elongated",
        "waist_shape": "no_waist"
    }
}


def infer_insect_from_filename(img_path):
    name = Path(img_path).stem.lower()

    for species_key, insect_label in SPECIES_TO_INSECT.items():
        if species_key in name:
            return insect_label

    return None


records = []

for split in ["train", "valid", "test"]:
    images_dir = DATASET_ROOT / split / "images"

    if not images_dir.exists():
        print("No existe:", images_dir)
        continue

    for img_path in images_dir.glob("*"):
        if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            continue

        insect_label = infer_insect_from_filename(img_path)

        if insect_label is None:
            records.append({
                "split": split,
                "image_path": str(img_path),
                "matched": False
            })
            continue

        item = {
            "split": split,
            "image_path": str(img_path),
            "matched": True,
            "insect_label": insect_label
        }

        for trait in TRAIT_COLUMNS:
            item[trait] = TRAITS_BY_INSECT[insect_label][trait]

        records.append(item)

mobile_df = pd.DataFrame(records)

print("Total imágenes encontradas:", len(mobile_df))
print("Imágenes con label:", mobile_df["matched"].sum())
print("Imágenes sin label:", (~mobile_df["matched"]).sum())

if (~mobile_df["matched"]).sum() > 0:
    print("\nEjemplos sin label:")
    display(mobile_df[mobile_df["matched"] == False].head(10))

mobile_df = mobile_df[mobile_df["matched"] == True].copy()

print("\nDistribución por split:")
print(mobile_df["split"].value_counts())

print("\nDistribución por insecto:")
print(mobile_df["insect_label"].value_counts())

display(mobile_df.head())

Total imágenes encontradas: 331
Imágenes con label: 331
Imágenes sin label: 0

Distribución por split:
split
train    231
valid     66
test      34
Name: count, dtype: int64

Distribución por insecto:
insect_label
sevenspotted_lady_beetle    58
carolina_grasshopper        57
european_mantid             56
honey_bee                   55
asian_tiger_mosquito        55
monarch_butterfly           50
Name: count, dtype: int64


,split,image_path,matched,insect_label,wing_count,forewing_type,mouthpart_type,antenna_type,leg_specialization,body_shape,waist_shape
0,train,/content/insect_dataset/insect_parts_dataset_c...,True,carolina_grasshopper,two_pairs,tegmina,chewing,filiform,jumping,elongated,no_waist
1,train,/content/insect_dataset/insect_parts_dataset_c...,True,honey_bee,two_pairs,membranous,chewing,filiform,generalist,elongated,narrow_waist
2,train,/content/insect_dataset/insect_parts_dataset_c...,True,honey_bee,two_pairs,membranous,chewing,filiform,generalist,elongated,narrow_waist
3,train,/content/insect_dataset/insect_parts_dataset_c...,True,monarch_butterfly,two_pairs,scaly,siphoning,clavate,generalist,slender,no_waist
4,train,/content/insect_dataset/insect_parts_dataset_c...,True,honey_bee,two_pairs,membranous,chewing,filiform,generalist,elongated,narrow_waist


In [5]:
OUTPUT_COLUMNS = ["insect_label"] + TRAIT_COLUMNS

label_maps = {}

for col in OUTPUT_COLUMNS:
    values = sorted(mobile_df[col].dropna().astype(str).unique().tolist())
    label_maps[col] = {value: idx for idx, value in enumerate(values)}

idx_maps = {
    col: {idx: value for value, idx in mapping.items()}
    for col, mapping in label_maps.items()
}

print(json.dumps(label_maps, indent=2))

{
  "insect_label": {
    "asian_tiger_mosquito": 0,
    "carolina_grasshopper": 1,
    "european_mantid": 2,
    "honey_bee": 3,
    "monarch_butterfly": 4,
    "sevenspotted_lady_beetle": 5
  },
  "wing_count": {
    "one_pair": 0,
    "two_pairs": 1
  },
  "forewing_type": {
    "elytra": 0,
    "membranous": 1,
    "scaly": 2,
    "tegmina": 3
  },
  "mouthpart_type": {
    "chewing": 0,
    "piercing_sucking": 1,
    "siphoning": 2
  },
  "antenna_type": {
    "clavate": 0,
    "filiform": 1
  },
  "leg_specialization": {
    "generalist": 0,
    "grasping": 1,
    "jumping": 2
  },
  "body_shape": {
    "elongated": 0,
    "rounded_domed": 1,
    "slender": 2
  },
  "waist_shape": {
    "narrow_waist": 0,
    "no_waist": 1
  }
}


In [6]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE_MOBILENET, IMG_SIZE_MOBILENET)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE_MOBILENET, IMG_SIZE_MOBILENET)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class InsectMultiTaskDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["image_path"]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        targets = {}
        for col in OUTPUT_COLUMNS:
            targets[col] = torch.tensor(
                label_maps[col][str(row[col])],
                dtype=torch.long
            )

        return image, targets


train_df = mobile_df[mobile_df["split"] == "train"].copy()
valid_df = mobile_df[mobile_df["split"] == "valid"].copy()
test_df = mobile_df[mobile_df["split"] == "test"].copy()

train_dataset = InsectMultiTaskDataset(train_df, transform=train_transform)
valid_dataset = InsectMultiTaskDataset(valid_df, transform=eval_transform)
test_dataset = InsectMultiTaskDataset(test_df, transform=eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train:", len(train_dataset))
print("Valid:", len(valid_dataset))
print("Test:", len(test_dataset))

Train: 231
Valid: 66
Test: 34


In [7]:
class MultiHeadMobileNet(nn.Module):
    def __init__(self, label_maps):
        super().__init__()

        weights = MobileNet_V3_Small_Weights.DEFAULT
        self.backbone = models.mobilenet_v3_small(weights=weights)

        in_features = self.backbone.classifier[0].in_features
        self.backbone.classifier = nn.Identity()

        self.dropout = nn.Dropout(0.25)

        self.heads = nn.ModuleDict()
        for col, mapping in label_maps.items():
            self.heads[col] = nn.Linear(in_features, len(mapping))

    def forward(self, x):
        features = self.backbone(x)
        features = self.dropout(features)

        outputs = {}
        for col, head in self.heads.items():
            outputs[col] = head(features)

        return outputs


mobilenet_model = MultiHeadMobileNet(label_maps).to(device)

criterions = {
    col: nn.CrossEntropyLoss()
    for col in OUTPUT_COLUMNS
}

optimizer = optim.AdamW(
    mobilenet_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

print("MobileNet multi-head creado correctamente.")

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 35.3MB/s]


MobileNet multi-head creado correctamente.


In [8]:
def train_one_epoch(model, loader):
    model.train()

    total_loss = 0
    correct = {col: 0 for col in OUTPUT_COLUMNS}
    total = 0

    for images, targets in loader:
        images = images.to(device)
        targets = {k: v.to(device) for k, v in targets.items()}

        optimizer.zero_grad()

        outputs = model(images)

        loss = 0
        for col in OUTPUT_COLUMNS:
            loss += criterions[col](outputs[col], targets[col])

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        batch_size = images.size(0)
        total += batch_size

        for col in OUTPUT_COLUMNS:
            preds = outputs[col].argmax(dim=1)
            correct[col] += (preds == targets[col]).sum().item()

    accs = {
        col: correct[col] / total if total > 0 else 0
        for col in OUTPUT_COLUMNS
    }

    avg_loss = total_loss / len(loader) if len(loader) > 0 else 0

    return avg_loss, accs


def evaluate(model, loader):
    model.eval()

    total_loss = 0
    correct = {col: 0 for col in OUTPUT_COLUMNS}
    total = 0

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            targets = {k: v.to(device) for k, v in targets.items()}

            outputs = model(images)

            loss = 0
            for col in OUTPUT_COLUMNS:
                loss += criterions[col](outputs[col], targets[col])

            total_loss += loss.item()

            batch_size = images.size(0)
            total += batch_size

            for col in OUTPUT_COLUMNS:
                preds = outputs[col].argmax(dim=1)
                correct[col] += (preds == targets[col]).sum().item()

    accs = {
        col: correct[col] / total if total > 0 else 0
        for col in OUTPUT_COLUMNS
    }

    avg_loss = total_loss / len(loader) if len(loader) > 0 else 0

    return avg_loss, accs

In [9]:
BEST_MOBILENET_PATH = Path("/content/best_mobilenet_insect_multitask.pth")

history = []
best_valid_insect_acc = -1

if TRAIN_MOBILENET:
    for epoch in range(1, MOBILENET_EPOCHS + 1):
        train_loss, train_accs = train_one_epoch(mobilenet_model, train_loader)
        valid_loss, valid_accs = evaluate(mobilenet_model, valid_loader)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "valid_loss": valid_loss,
            "train_insect_acc": train_accs["insect_label"],
            "valid_insect_acc": valid_accs["insect_label"]
        }

        for col in TRAIT_COLUMNS:
            row[f"valid_{col}_acc"] = valid_accs[col]

        history.append(row)

        print(f"\nEpoch {epoch}/{MOBILENET_EPOCHS}")
        print(f"Train Loss: {train_loss:.4f} | Valid Loss: {valid_loss:.4f}")
        print(f"Train insect acc: {train_accs['insect_label']:.3f}")
        print(f"Valid insect acc: {valid_accs['insect_label']:.3f}")

        print("Valid traits:")
        for col in TRAIT_COLUMNS:
            print(f"  {col}: {valid_accs[col]:.3f}")

        if valid_accs["insect_label"] > best_valid_insect_acc:
            best_valid_insect_acc = valid_accs["insect_label"]
            torch.save(mobilenet_model.state_dict(), BEST_MOBILENET_PATH)
            print("Nuevo mejor modelo guardado.")

else:
    BEST_MOBILENET_PATH = FINAL_MODELS_DIR / "best_mobilenet_insect_multitask.pth"

if not BEST_MOBILENET_PATH.exists():
    raise FileNotFoundError(f"No se encontró MobileNet en: {BEST_MOBILENET_PATH}")

print("\nMejor accuracy de insecto en validación:", best_valid_insect_acc)
print("Modelo guardado en:", BEST_MOBILENET_PATH)


Epoch 1/15
Train Loss: 8.3569 | Valid Loss: 7.7788
Train insect acc: 0.247
Valid insect acc: 0.227
Valid traits:
  wing_count: 0.833
  forewing_type: 0.318
  mouthpart_type: 0.303
  antenna_type: 0.727
  leg_specialization: 0.682
  body_shape: 0.439
  waist_shape: 0.848
Nuevo mejor modelo guardado.

Epoch 2/15
Train Loss: 7.3416 | Valid Loss: 6.9208
Train insect acc: 0.359
Valid insect acc: 0.348
Valid traits:
  wing_count: 0.833
  forewing_type: 0.379
  mouthpart_type: 0.712
  antenna_type: 0.727
  leg_specialization: 0.667
  body_shape: 0.545
  waist_shape: 0.833
Nuevo mejor modelo guardado.

Epoch 3/15
Train Loss: 6.5466 | Valid Loss: 6.2479
Train insect acc: 0.528
Valid insect acc: 0.485
Valid traits:
  wing_count: 0.833
  forewing_type: 0.455
  mouthpart_type: 0.773
  antenna_type: 0.788
  leg_specialization: 0.667
  body_shape: 0.667
  waist_shape: 0.833
Nuevo mejor modelo guardado.

Epoch 4/15
Train Loss: 5.9208 | Valid Loss: 5.4402
Train insect acc: 0.654
Valid insect acc: 0.6

In [10]:
mobilenet_model.load_state_dict(
    torch.load(BEST_MOBILENET_PATH, map_location=device)
)

mobilenet_model.eval()

test_loss, test_accs = evaluate(mobilenet_model, test_loader)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test insect acc: {test_accs['insect_label']:.3f}")

print("\nTest traits:")
for col in TRAIT_COLUMNS:
    print(f"{col}: {test_accs[col]:.3f}")

Test Loss: 0.6137
Test insect acc: 0.971

Test traits:
wing_count: 0.971
forewing_type: 1.000
mouthpart_type: 0.971
antenna_type: 1.000
leg_specialization: 0.971
body_shape: 0.971
waist_shape: 1.000


In [11]:
def safe_copy(src, dst):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)

    if not src.exists():
        raise FileNotFoundError(f"No existe el archivo origen: {src}")

    if dst.exists() and src.resolve() == dst.resolve():
        print("Ya está en el destino:", dst)
        return dst

    shutil.copy2(src, dst)
    print("Copiado:", dst)
    return dst


# Guardar modelo
MOBILENET_FINAL_PATH = FINAL_MODELS_DIR / "best_mobilenet_insect_multitask.pth"
safe_copy(BEST_MOBILENET_PATH, MOBILENET_FINAL_PATH)

# Guardar mapas
with open(FINAL_MODELS_DIR / "label_maps.json", "w", encoding="utf-8") as f:
    json.dump(label_maps, f, indent=2, ensure_ascii=False)

with open(FINAL_MODELS_DIR / "idx_maps.json", "w", encoding="utf-8") as f:
    json.dump(idx_maps, f, indent=2, ensure_ascii=False)

# Guardar índice usado para MobileNet
mobile_df.to_csv(FINAL_MODELS_DIR / "mobilenet_dataset_index.csv", index=False)

# Guardar historial
history_df = pd.DataFrame(history)
history_df.to_csv(FINAL_MODELS_DIR / "mobilenet_training_history.csv", index=False)

# Guardar resumen
summary = {
    "mobilenet_epochs": MOBILENET_EPOCHS,
    "batch_size": BATCH_SIZE,
    "image_size": IMG_SIZE_MOBILENET,
    "train_images": len(train_dataset),
    "valid_images": len(valid_dataset),
    "test_images": len(test_dataset),
    "best_valid_insect_accuracy": best_valid_insect_acc,
    "test_loss": test_loss,
    "test_accuracies": test_accs,
    "output_columns": OUTPUT_COLUMNS,
    "trait_columns": TRAIT_COLUMNS
}

with open(FINAL_MODELS_DIR / "mobilenet_training_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

# Backup ZIP de todos los modelos finales
backup_path = "/content/final_models_backup"
shutil.make_archive(backup_path, "zip", FINAL_MODELS_DIR)

safe_copy(
    Path(backup_path + ".zip"),
    DRIVE_PROJECT_DIR / "final_models_backup.zip"
)

print("\nMobileNet guardado correctamente.")
print("Modelo MobileNet final:", MOBILENET_FINAL_PATH)
print("label_maps:", FINAL_MODELS_DIR / "label_maps.json")
print("idx_maps:", FINAL_MODELS_DIR / "idx_maps.json")
print("Backup:", DRIVE_PROJECT_DIR / "final_models_backup.zip")

Copiado: /content/drive/MyDrive/ModeloEntrenado/final_models/best_mobilenet_insect_multitask.pth
Copiado: /content/drive/MyDrive/ModeloEntrenado/final_models_backup.zip

MobileNet guardado correctamente.
Modelo MobileNet final: /content/drive/MyDrive/ModeloEntrenado/final_models/best_mobilenet_insect_multitask.pth
label_maps: /content/drive/MyDrive/ModeloEntrenado/final_models/label_maps.json
idx_maps: /content/drive/MyDrive/ModeloEntrenado/final_models/idx_maps.json
Backup: /content/drive/MyDrive/ModeloEntrenado/final_models_backup.zip


In [12]:
from pathlib import Path
import json
import pandas as pd

# =========================
# GUARDAR MÉTRICAS DE MOBILENET
# =========================

FINAL_MODELS_DIR = Path("/content/drive/MyDrive/ModeloEntrenado/final_models")

mobilenet_metrics = {
    "model": "MobileNetV3 Small multi-head",
    "dataset_counts": {
        "train": len(train_dataset),
        "valid": len(valid_dataset),
        "test": len(test_dataset)
    },
    "best_valid_insect_accuracy": best_valid_insect_acc,
    "test_loss": test_loss,
    "test_accuracies": test_accs
}

metrics_json_path = FINAL_MODELS_DIR / "mobilenet_metrics.json"

with open(metrics_json_path, "w") as f:
    json.dump(mobilenet_metrics, f, indent=2)

print("Métricas MobileNet guardadas en:")
print(metrics_json_path)

# Guardar historial de entrenamiento
history_df = pd.DataFrame(history)
history_csv_path = FINAL_MODELS_DIR / "mobilenet_training_history.csv"
history_df.to_csv(history_csv_path, index=False)

print("\nHistorial de entrenamiento guardado en:")
print(history_csv_path)

display(history_df.tail())

Métricas MobileNet guardadas en:
/content/drive/MyDrive/ModeloEntrenado/final_models/mobilenet_metrics.json

Historial de entrenamiento guardado en:
/content/drive/MyDrive/ModeloEntrenado/final_models/mobilenet_training_history.csv


,epoch,train_loss,valid_loss,train_insect_acc,valid_insect_acc,valid_wing_count_acc,valid_forewing_type_acc,valid_mouthpart_type_acc,valid_antenna_type_acc,valid_leg_specialization_acc,valid_body_shape_acc,valid_waist_shape_acc
10,11,1.588983,1.468147,0.948052,0.909091,0.954545,0.939394,0.969697,1.0,0.909091,0.969697,1.0
11,12,1.427928,1.255350,0.952381,0.939394,0.954545,0.939394,0.969697,1.0,0.909091,0.984848,1.0
12,13,1.117423,1.090577,0.969697,0.954545,0.954545,0.969697,0.984848,1.0,0.909091,0.984848,1.0
13,14,0.999430,1.031526,0.948052,0.939394,0.954545,0.954545,0.984848,1.0,0.909091,0.984848,1.0
14,15,0.821676,0.986274,0.969697,0.969697,0.954545,0.954545,0.984848,1.0,0.909091,0.984848,1.0
